# EDA of the new tables for ML: hh_demographic, causal_data, coupon_redempt, product

**Purpose**: check the shape, missing values and duplicates of the new tables to be used in the "top 20% decline prediction model".
(transaction_data was already examined enough in the earlier notebooks, so only the 4 newly used tables are covered here)

**Note**: every feature computation must use only **early-window (weeks 17-26)** information (to prevent data leakage),
so where possible this notebook also checks the version filtered to the early window.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

candidates = ["Malgun Gothic", "NanumGothic", "AppleGothic"]
available = {f.name for f in fm.fontManager.ttflist}
font_name = next((c for c in candidates if c in available), None)
if font_name:
    plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)

from config import DATA_DIR, OUT_DIR, INTERMEDIATE_DIR, FIG_DIR, SEED

FILES = {
    "hh_demographic": "hh_demographic.csv",
    "causal_data":    "causal_data.csv",
    "coupon_redempt": "coupon_redempt.csv",
    "product":        "product.csv",
}

dfs = {name: pd.read_csv(DATA_DIR / fname) for name, fname in FILES.items()}

for name, df in dfs.items():
    print(f"{name}: {df.shape}")


hh_demographic: (801, 8)
causal_data: (36786524, 5)
coupon_redempt: (2318, 4)
product: (92353, 7)


## 1. Detailed look at each table (shape, dtypes, missing values, duplicates, categorical values)

In [2]:
table_checks = {}


def inspect(name):
    df = dfs[name]
    print(f"===== {name}  (shape: {df.shape}) =====\n")

    print("--- head(3) ---")
    display(df.head(3))

    print("\n--- dtypes ---")
    print(df.dtypes)

    n_dup = df.duplicated().sum()
    table_checks[name] = {"rows": len(df), "cols": df.shape[1], "duplicate_rows": int(n_dup),
                          "missing_cells": int(df.isna().sum().sum())}
    print(f"\n--- Fully duplicated rows: {n_dup:,} ({n_dup/len(df)*100:.3f}%) ---")

    print("\n--- Missing share (%) ---")
    na_ratio = (df.isna().sum() / len(df) * 100).round(2)
    print(na_ratio[na_ratio > 0] if na_ratio.sum() > 0 else "No missing values")

    num_cols = df.select_dtypes(include=np.number).columns
    if len(num_cols) > 0:
        print("\n--- Numeric describe ---")
        display(df[num_cols].describe().T)

    cat_cols = df.select_dtypes(include=['object', 'str']).columns
    for c in cat_cols:
        nunique = df[c].nunique()
        print(f"\n--- '{c}': {nunique} unique values, top 10 ---")
        print(df[c].value_counts().head(10))

    print("\n")


In [3]:
inspect("hh_demographic")

===== hh_demographic  (shape: (801, 8)) =====

--- head(3) ---


,AGE_DESC,MARITAL_STATUS_CODE,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC,household_key
0,65+,A,35-49K,Homeowner,2 Adults No Kids,2,None/Unknown,1
1,45-54,A,50-74K,Homeowner,2 Adults No Kids,2,None/Unknown,7
2,25-34,U,25-34K,Unknown,2 Adults Kids,3,1,8



--- dtypes ---
AGE_DESC                 str
MARITAL_STATUS_CODE      str
INCOME_DESC              str
HOMEOWNER_DESC           str
HH_COMP_DESC             str
HOUSEHOLD_SIZE_DESC      str
KID_CATEGORY_DESC        str
household_key          int64
dtype: object

--- Fully duplicated rows: 0 (0.000%) ---

--- Missing share (%) ---
No missing values

--- Numeric describe ---


,count,mean,std,min,25%,50%,75%,max
household_key,801.0,1235.17603,736.804647,1.0,596.0,1218.0,1914.0,2499.0



--- 'AGE_DESC': 6 unique values, top 10 ---
AGE_DESC
45-54    288
35-44    194
25-34    142
65+       72
55-64     59
19-24     46
Name: count, dtype: int64

--- 'MARITAL_STATUS_CODE': 3 unique values, top 10 ---
MARITAL_STATUS_CODE
U    344
A    340
B    117
Name: count, dtype: int64

--- 'INCOME_DESC': 12 unique values, top 10 ---
INCOME_DESC
50-74K       192
35-49K       172
75-99K        96
25-34K        77
15-24K        74
Under 15K     61
125-149K      38
100-124K      34
150-174K      30
250K+         11
Name: count, dtype: int64

--- 'HOMEOWNER_DESC': 5 unique values, top 10 ---
HOMEOWNER_DESC
Homeowner          504
Unknown            233
Renter              42
Probable Renter     11
Probable Owner      11
Name: count, dtype: int64

--- 'HH_COMP_DESC': 6 unique values, top 10 ---
HH_COMP_DESC
2 Adults No Kids    255
2 Adults Kids       187
Single Female       144
Single Male          95
Unknown              73
1 Adult Kids         47
Name: count, dtype: int64

--- 'HOUSEHOLD_S

In [4]:
inspect("causal_data")

===== causal_data  (shape: (36786524, 5)) =====

--- head(3) ---


,PRODUCT_ID,STORE_ID,WEEK_NO,display,mailer
0,26190,286,70,0,A
1,26190,288,70,0,A
2,26190,289,70,0,A



--- dtypes ---
PRODUCT_ID    int64
STORE_ID      int64
WEEK_NO       int64
display         str
mailer          str
dtype: object



--- Fully duplicated rows: 0 (0.000%) ---

--- Missing share (%) ---


No missing values

--- Numeric describe ---


,count,mean,std,min,25%,50%,75%,max
PRODUCT_ID,36786524.0,3.512237e+06,4.094819e+06,26190.0,928850.0,1312148.0,5730125.0,18244453.0
STORE_ID,36786524.0,3.234151e+03,9.106904e+03,286.0,329.0,369.0,421.0,34280.0
WEEK_NO,36786524.0,5.529931e+01,2.696101e+01,9.0,32.0,56.0,78.0,101.0



--- 'display': 10 unique values, top 10 ---


display
0    21038745
9     2699467
5     2575289
7     2362118
3     2073738
6     1816021
2     1812840
1     1102141
A      713180
4      592985
Name: count, dtype: int64



--- 'mailer': 11 unique values, top 10 ---


mailer
A    17106789
0    11534183
D     4467453
H     1560395
F     1077549
J      306924
L      301327
C      291059
X      120823
Z       19453
Name: count, dtype: int64




In [5]:
inspect("coupon_redempt")

===== coupon_redempt  (shape: (2318, 4)) =====

--- head(3) ---


,household_key,DAY,COUPON_UPC,CAMPAIGN
0,1,421,10000085364,8
1,1,421,51700010076,8
2,1,427,54200000033,8



--- dtypes ---
household_key    int64
DAY              int64
COUPON_UPC       int64
CAMPAIGN         int64
dtype: object

--- Fully duplicated rows: 0 (0.000%) ---

--- Missing share (%) ---
No missing values

--- Numeric describe ---


,count,mean,std,min,25%,50%,75%,max
household_key,2318.0,1.302817e+03,7.830025e+02,1.000000e+00,5.880000e+02,1.396500e+03,2.004000e+03,2.500000e+03
DAY,2318.0,5.282174e+02,1.003610e+02,2.250000e+02,4.582500e+02,5.320000e+02,6.050000e+02,7.040000e+02
COUPON_UPC,2318.0,4.123049e+10,1.986068e+10,1.000009e+10,1.000009e+10,5.234003e+10,5.430002e+10,5.897850e+10
CAMPAIGN,2318.0,1.555134e+01,5.716636e+00,1.000000e+00,1.300000e+01,1.400000e+01,1.800000e+01,3.000000e+01


In [6]:
inspect("product")

===== product  (shape: (92353, 7)) =====

--- head(3) ---


,PRODUCT_ID,MANUFACTURER,DEPARTMENT,BRAND,COMMODITY_DESC,SUB_COMMODITY_DESC,CURR_SIZE_OF_PRODUCT
0,25671,2,GROCERY,National,FRZN ICE,ICE - CRUSHED/CUBED,22 LB
1,26081,2,MISC. TRANS.,National,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,
2,26093,69,PASTRY,Private,BREAD,BREAD:ITALIAN/FRENCH,



--- dtypes ---
PRODUCT_ID              int64
MANUFACTURER            int64
DEPARTMENT                str
BRAND                     str
COMMODITY_DESC            str
SUB_COMMODITY_DESC        str
CURR_SIZE_OF_PRODUCT      str
dtype: object

--- Fully duplicated rows: 0 (0.000%) ---

--- Missing share (%) ---
No missing values

--- Numeric describe ---


,count,mean,std,min,25%,50%,75%,max
PRODUCT_ID,92353.0,5.328353e+06,5.359937e+06,25671.0,970628.0,1621091.0,9704770.0,18316298.0
MANUFACTURER,92353.0,1.739228e+03,1.818270e+03,1.0,328.0,1094.0,2264.0,6477.0



--- 'DEPARTMENT': 44 unique values, top 10 ---
DEPARTMENT
GROCERY       39021
DRUG GM       31529
PRODUCE        3118
COSMETICS      3011
NUTRITION      2914
MEAT           2544
MEAT-PCKGD     2427
DELI           2354
PASTRY         2149
FLORAL          938
Name: count, dtype: int64

--- 'BRAND': 2 unique values, top 10 ---
BRAND
National    78537
Private     13816
Name: count, dtype: int64

--- 'COMMODITY_DESC': 308 unique values, top 10 ---
COMMODITY_DESC
GREETING CARDS/WRAP/PARTY SPLY    2785
CANDY - PACKAGED                  2475
MAKEUP AND TREATMENT              2467
HAIR CARE PRODUCTS                1744
SOFT DRINKS                       1704
BAG SNACKS                        1523
HISPANIC                          1460
FRZN MEAT/MEAT DINNERS            1268
STATIONERY & SCHOOL SUPPLIES      1261
MAGAZINE                          1224
Name: count, dtype: int64

--- 'SUB_COMMODITY_DESC': 2383 unique values, top 10 ---
SUB_COMMODITY_DESC
CARDS EVERYDAY            1005
BEERALEMALT L

## 2. hh_demographic: overall coverage by household_key

The share of stable-window households that have demographic information. (Re-check that it matches the 32% seen in the earlier overview of the 8 tables)

In [7]:
tx = pd.read_csv(DATA_DIR / "transaction_data.csv")

STABLE_MIN_WEEK = 17
STABLE_MAX_WEEK = 99
EARLY_N = 10

tx_stable = tx[(tx["WEEK_NO"] >= STABLE_MIN_WEEK) & (tx["WEEK_NO"] <= STABLE_MAX_WEEK)].copy()
stable_households = np.sort(tx_stable["household_key"].unique())

demo_hh = set(dfs["hh_demographic"]["household_key"].unique())
stable_hh_set = set(stable_households)

coverage = len(stable_hh_set & demo_hh) / len(stable_hh_set) * 100
print(f"Share of all stable-window households present in hh_demographic: {coverage:.1f}%")


Share of all stable-window households present in hh_demographic: 32.1%


## 3. hh_demographic: re-check coverage for only the early top 20% households (499)

The top 20% households that become the model-training population may differ from the overall average (32%), so check them separately.

In [8]:
early_window = tx_stable[
    (tx_stable["WEEK_NO"] >= STABLE_MIN_WEEK) &
    (tx_stable["WEEK_NO"] < STABLE_MIN_WEEK + EARLY_N)
]
sales_early_full = early_window.groupby("household_key")["SALES_VALUE"].sum()

panel_wide = pd.DataFrame(index=stable_households)
panel_wide["sales_early"] = sales_early_full.reindex(panel_wide.index).fillna(0)

cutoff_value = panel_wide["sales_early"].quantile(0.8)
panel_wide["group"] = np.where(panel_wide["sales_early"] >= cutoff_value, "top 20%", "remaining 80%")

top20_households = set(panel_wide[panel_wide["group"] == "top 20%"].index)
print(f"Top 20% households on the early-window basis: {len(top20_households):,}")

top20_coverage = len(top20_households & demo_hh) / len(top20_households) * 100
print(f"Share of top 20% households present in hh_demographic: {top20_coverage:.1f}%")
print(f"-> Households that will be treated as missing (No record): {len(top20_households) - len(top20_households & demo_hh)}")


Top 20% households on the early-window basis: 499
Share of top 20% households present in hh_demographic: 63.9%
-> Households that will be treated as missing (No record): 180


## 4. causal_data: size when filtered to the early window (weeks 17-26) only

Check the full size together with the size when keeping only the early window actually used for the feature computation (also a leakage check).

In [9]:
print(dfs["causal_data"].columns.tolist())

['PRODUCT_ID', 'STORE_ID', 'WEEK_NO', 'display', 'mailer']


In [10]:
causal_early = dfs["causal_data"][
    (dfs["causal_data"]["WEEK_NO"] >= STABLE_MIN_WEEK) &
    (dfs["causal_data"]["WEEK_NO"] < STABLE_MIN_WEEK + EARLY_N)
]

print(f"causal_data total rows: {len(dfs['causal_data']):,}")
print(f"After filtering to the early window (weeks 17-26) only: {len(causal_early):,} ({len(causal_early)/len(dfs['causal_data'])*100:.1f}%)")

print("\nDISPLAY value distribution (early window)")
print(causal_early["display"].value_counts())

print("\nMAILER value distribution (early window)")
print(causal_early["mailer"].value_counts())


causal_data total rows: 36,786,524
After filtering to the early window (weeks 17-26) only: 3,844,057 (10.4%)

DISPLAY value distribution (early window)
display
0    2095870
9     293258
5     290242
7     251627
3     230012
6     208397
2     203205
1     123593
A      80146
4      67707
Name: count, dtype: int64

MAILER value distribution (early window)
mailer
A    1814795
0    1288246
D     372445
H     189788
F      93230
L      43977
C      22659
J      18917
Name: count, dtype: int64


## 5. coupon_redempt: size when filtered to the early window (by DAY) only

coupon_redempt has no WEEK_NO, only a DAY column, so convert weeks to days before filtering.

In [11]:
early_day_min = tx.loc[tx["WEEK_NO"] == STABLE_MIN_WEEK, "DAY"].min()
early_day_max = tx.loc[tx["WEEK_NO"] == STABLE_MIN_WEEK + EARLY_N - 1, "DAY"].max()
print(f"Early-window DAY range: {early_day_min} to {early_day_max}")

coupon_redempt_early = dfs["coupon_redempt"][
    (dfs["coupon_redempt"]["DAY"] >= early_day_min) &
    (dfs["coupon_redempt"]["DAY"] <= early_day_max)
]

print(f"\ncoupon_redempt total rows: {len(dfs['coupon_redempt']):,}")
print(f"After filtering to the early window only: {len(coupon_redempt_early):,} ({len(coupon_redempt_early)/len(dfs['coupon_redempt'])*100:.1f}%)")

# Share of top 20% households with a coupon redemption in the early window
redeemers_early = set(coupon_redempt_early["household_key"].unique())
top20_redeemer_pct = len(top20_households & redeemers_early) / len(top20_households) * 100
print(f"\nShare of top 20% households with a coupon redemption in the early window: {top20_redeemer_pct:.1f}%")
print("-> If this share is low (a sparse feature), it is safer to use it as a binary (0/1) variable in the model")


Early-window DAY range: 111 to 180

coupon_redempt total rows: 2,318
After filtering to the early window only: 0 (0.0%)

Share of top 20% households with a coupon redemption in the early window: 0.0%
-> If this share is low (a sparse feature), it is safer to use it as a binary (0/1) variable in the model


## 6. product: check the join keys with causal_data / transaction_data

Check for duplicate product_id and missing values (whether joining causes problems such as many-to-many matches).

In [12]:
print(f"Unique product_id: {dfs['product']['PRODUCT_ID'].nunique():,}")
print(f"Total rows: {len(dfs['product']):,}")
print(f"Duplicate product_id: {'yes, needs checking' if dfs['product']['PRODUCT_ID'].nunique() != len(dfs['product']) else 'none (1:1 mapping confirmed)'}")


Unique product_id: 92,353
Total rows: 92,353
Duplicate product_id: none (1:1 mapping confirmed)


## 7. Notes for the next step

- Missing hh_demographic is included as a "No record" category (check the actual missing size using the result of section 3)
- causal_data and coupon_redempt must be filtered to the early window only (week 17-26 / the corresponding DAY range) for the feature computation; reuse the filtering logic of sections 4 and 5 as is
- If the share of coupon_redempt redemption history is low (sparse), consider simplifying to "redeemed or not (0/1)" rather than "number of redemptions"

## 8. campaign_table / campaign_desc: were there campaigns in the early window?

Check whether there is the same problem as coupon_redempt having nothing in the early window (DAY 111-180),
and whether campaign sends themselves happened in this window.

In [13]:
dfs["campaign_table"] = pd.read_csv(DATA_DIR / "campaign_table.csv")
dfs["campaign_desc"] = pd.read_csv(DATA_DIR / "campaign_desc.csv")

for name in ["campaign_table", "campaign_desc"]:
    df = dfs[name]
    print(f"===== {name} (shape: {df.shape}) =====")
    display(df.head(3))
    print("dtypes:")
    print(df.dtypes)
    n_dup = df.duplicated().sum()
    table_checks[name] = {"rows": len(df), "cols": df.shape[1], "duplicate_rows": int(n_dup),
                          "missing_cells": int(df.isna().sum().sum())}
    print(f"Fully duplicated rows: {n_dup:,}")
    na_ratio = (df.isna().sum() / len(df) * 100).round(2)
    print("Missing share (%):")
    print(na_ratio[na_ratio > 0] if na_ratio.sum() > 0 else "No missing values")
    print()


===== campaign_table (shape: (7208, 3)) =====


,DESCRIPTION,household_key,CAMPAIGN
0,TypeA,17,26
1,TypeA,27,26
2,TypeA,212,26


dtypes:
DESCRIPTION        str
household_key    int64
CAMPAIGN         int64
dtype: object
Fully duplicated rows: 0
Missing share (%):
No missing values

===== campaign_desc (shape: (30, 4)) =====


,DESCRIPTION,CAMPAIGN,START_DAY,END_DAY
0,TypeB,24,659,719
1,TypeC,15,547,708
2,TypeB,25,659,691


dtypes:


DESCRIPTION      str
CAMPAIGN       int64
START_DAY      int64
END_DAY        int64
dtype: object
Fully duplicated rows: 0
Missing share (%):
No missing values



### 8-1. Distribution of campaign start days (START_DAY): do they overlap the early window (DAY 111-180)?

In [14]:
print("campaign_desc START_DAY / END_DAY summary")
print(dfs["campaign_desc"][["START_DAY", "END_DAY"]].describe())

early_day_min = 111  # reuse the early-window DAY range confirmed earlier
early_day_max = 180

# Campaigns whose period overlaps the early window at all (starts before the early window ends and ends after it starts)
overlapping_campaigns = dfs["campaign_desc"][
    (dfs["campaign_desc"]["START_DAY"] <= early_day_max) &
    (dfs["campaign_desc"]["END_DAY"] >= early_day_min)
]

print(f"\nCampaigns overlapping the early window (DAY {early_day_min}-{early_day_max}): {len(overlapping_campaigns)} / {len(dfs['campaign_desc'])} in total")
display(overlapping_campaigns)


campaign_desc START_DAY / END_DAY summary
        START_DAY     END_DAY
count   30.000000   30.000000
mean   463.866667  510.466667
std    134.488490  137.730555
min    224.000000  264.000000
25%    360.000000  405.750000
50%    470.000000  502.000000
75%    584.000000  640.250000
max    659.000000  719.000000

Campaigns overlapping the early window (DAY 111-180): 0 / 30 in total


,DESCRIPTION,CAMPAIGN,START_DAY,END_DAY


### 8-2. Among the top 20% households, the share that received a campaign overlapping the early window

In [15]:
if len(overlapping_campaigns) > 0:
    overlapping_campaign_ids = overlapping_campaigns["CAMPAIGN"].unique()

    campaign_table_early = dfs["campaign_table"][
        dfs["campaign_table"]["CAMPAIGN"].isin(overlapping_campaign_ids)
    ]

    print(f"Send records of campaigns overlapping the early window: {len(campaign_table_early):,}")

    recipients_early = set(campaign_table_early["household_key"].unique())
    top20_campaign_pct = len(top20_households & recipients_early) / len(top20_households) * 100

    print(f"Share of top 20% households that received a campaign in the early window: {top20_campaign_pct:.1f}%")

    all_recipients_pct = len(stable_hh_set & recipients_early) / len(stable_hh_set) * 100
    print(f"Share of all stable-window households that received a campaign in the early window: {all_recipients_pct:.1f}%")
else:
    print("No campaign overlaps the early window -> this feature cannot be used either; consider excluding it like coupon_redempt")


No campaign overlaps the early window -> this feature cannot be used either; consider excluding it like coupon_redempt


In [16]:
# --- source CSVs: source-table checks and the facts quoted in the README
pd.DataFrame(table_checks).T.rename_axis("table").reset_index().to_csv(OUT_DIR / "ml_source_table_checks.csv", index=False)
_facts = [
    ("households_stable_window", len(stable_hh_set)),
    ("hh_demographic_households", len(demo_hh)),
    ("demo_coverage_stable_pct", coverage),
    ("n_top20_early", len(top20_households)),
    ("top20_with_demo", len(top20_households & demo_hh)),
    ("top20_without_demo", len(top20_households) - len(top20_households & demo_hh)),
    ("top20_demo_coverage_pct", top20_coverage),
    ("top20_without_demo_pct", 100 - top20_coverage),
    ("coupon_redempt_min_day", dfs["coupon_redempt"]["DAY"].min()),
    ("early_day_min", early_day_min),
    ("early_day_max", early_day_max),
    ("coupon_redempt_rows_in_early_window", len(coupon_redempt_early)),
    ("campaigns_total", len(dfs["campaign_desc"])),
    ("campaigns_overlapping_early_window", len(overlapping_campaigns)),
    ("product_rows", len(dfs["product"])),
    ("product_unique_ids", dfs["product"]["PRODUCT_ID"].nunique()),
]
pd.DataFrame(_facts, columns=["metric", "value"]).to_csv(OUT_DIR / "ml_source_table_facts.csv", index=False)

### 8-3. Judgment

- If 8-1 shows 0 overlapping campaigns -> campaign-related features cannot be used in the early window, for the same reason as coupon_redempt
- If there are overlapping campaigns and the two shares of 8-2 (top 20% vs all) differ -> "whether a campaign was received in the early window (0/1)" can be adopted as a feature
- It could be split further by campaign type (Type A/B/C), but given the sample size (499 households), simplifying to receipt (binary) first is recommended